# 03 Abnormal returns and outcomes

Market model of every index, fitted on a window placed around every trading day, and from it: the normal days of
each country with their abnormal returns (AR), the thresholds of the binary outcome, and AR, SAR, outcomes and
pre-event volatility of every event row of `events.csv`. Two models: the main one, and the external factor alone
(robustness).

In [ ]:
import sys
from pathlib import Path

# Folder of helpers.py: the notebook's own folder or, on Nuvolos (where the kernel starts elsewhere), found by search.
# The search accepts only a code/ folder of THIS project (helpers.py, analysis.py and ../data/crises.csv) and stops
# if it finds more than one, so another project's helpers.py can never be imported by mistake.
def find_code_dir():
    found = set()
    for root in [Path.cwd(), Path("/files"), Path.home()]:
        for p in [root / "helpers.py", *root.glob("**/code/helpers.py")]:
            d = p.parent
            if p.exists() and (d / "analysis.py").exists() and (d.parent / "data" / "crises.csv").exists():
                found.add(d.resolve())
        if found:
            break
    if len(found) != 1:
        raise FileNotFoundError(f"expected one code/ folder of this project, found {sorted(map(str, found))}")
    return found.pop()


sys.path.insert(0, str(find_code_dir()))

import hashlib

import numpy as np
import pandas as pd

from helpers import (AR_WIN, EST_WIN, EXTERNAL_CANDIDATES, OUTCOME_DAYS, POL_EXCL, PRE_VOL_WIN, SEED, TICKERS,
                     load_derived, load_external, save_derived)
from analysis import ANALYSIS_GROUPS, MODELS, analysis_sample, event_outcomes, market_models, normal_days, thresholds

political_dates = load_derived("political_dates", parse_dates=["date", "trading_day"])
events = load_derived("events", parse_dates=["source_date", "day0", "day1"]).fillna({"ineligible_reason": ""})
# estimation window, pre-event window and outcome days do not overlap
assert EST_WIN[1] < AR_WIN[0] <= PRE_VOL_WIN[0] and PRE_VOL_WIN[1] < 0 <= min(OUTCOME_DAYS) <= max(OUTCOME_DAYS) <= AR_WIN[1]

## Returns, factors and market model
**Returns.** Log returns between consecutive valid closes of each index (the trading calendar of 02), in local
currency. A return spanning more than `LONG_GAP_DAYS` calendar days is a long-gap return. Israel has two day classes:
while TASE traded on Sunday, Yahoo has no Sunday close, so a Monday return spans two sessions.

**Factors**, over the same span as the country's return (from its previous valid close to the current one):
- g, the peers: equal-weighted mean of the log returns of the other 22 indices. Index j contributes from
  `GLOBAL_ENTRY_HISTORY` trading days after its first close, if its close at the start of the span is at most
  `GLOBAL_MAX_STALE_DAYS` calendar days old; with fewer than `MIN_GLOBAL` contributors g is missing.
- w, the external factor: log return of the first series in `EXTERNAL_CANDIDATES` that covers the whole period (the
  one used is printed below), with the same staleness rule.

**Market model.** r(t) = a + b0·g(t) + b1·g(t−1) + c0·w(t) + c1·w(t−1), fitted by OLS on the trading days `EST_WIN`
around the target day. The lags capture news that reaches an index on the next session (markets that close before
New York). Estimation days must have every regressor, must not be a long-gap return or the day after one (its
lagged factors span the gap), and must lie more than `POL_EXCL` trading days from every political date. A fit needs
`MIN_EST_DAYS` days; s is the residual sd with one degree of freedom per coefficient. The robustness model keeps only
the external factor: r(t) = a + c0·w(t) + c1·w(t−1). All windows are fitted at once from cumulative sums of the
cross-products; the check below refits 100 random windows directly.

In [ ]:
panels, fits = market_models(political_dates)
print("External factor:", EXTERNAL_CANDIDATES[load_external()[1]])

rng = np.random.default_rng(SEED)
for _ in range(100):
    c, m = rng.choice(list(TICKERS)), rng.choice(list(MODELS))
    p, f, x = panels[c], fits[m][c], list(MODELS[m])
    t = rng.choice(np.flatnonzero(f["s"].notna().to_numpy()))
    win = np.arange(max(t + EST_WIN[0], 0), t + EST_WIN[1] + 1)
    win = win[(p["est_ok"] & p[x].notna().all(axis=1)).to_numpy()[win]]
    X = np.column_stack([np.ones(len(win)), p[x].to_numpy()[win]])
    beta, ssr, *_ = np.linalg.lstsq(X, p["r"].to_numpy()[win], rcond=None)
    pol = p.index.get_indexer(political_dates.loc[political_dates["country"] == c, "trading_day"].dropna())
    assert len(win) == f["n_est"].iloc[t] and np.allclose(f[["a", *x]].iloc[t], beta, rtol=0, atol=1e-8)
    assert np.isclose(f["s"].iloc[t], np.sqrt(ssr[0] / (len(win) - X.shape[1])), rtol=1e-8)
    assert np.abs(win[:, None] - pol[None, :]).min() > POL_EXCL
print("Fast fits equal direct least squares on 100 random windows.")

## Normal days and thresholds
Every trading day from `EVENTS_FROM` to `SAMPLE_END` gets its one-day AR, with the window placed around that day.
A normal day has an AR, is not a long-gap return and lies more than `NORMAL_EXCL` trading days from every political
date. The thresholds of the binary outcome (robustness) are the quantiles `P_MAIN` and `P_SEC` of |AR| over the
normal days of each country and day class (main model).

The table describes the main model over the normal days of each country: median coefficients and R², and the
median R² of the external-only model.

In [ ]:
normal = {m: normal_days(panels, fits[m]) for m in MODELS}
assert all((normal[m][["country", "date"]] == normal["main"][["country", "date"]]).all().all() for m in MODELS)
thr = thresholds(normal["main"])

nd = normal["main"][normal["main"]["normal"]]
coef = pd.concat([fits["main"][c].loc[nd.loc[nd["country"] == c, "date"]].assign(country=c, r2_external=fits["external"][c]["r2"])
                  for c in TICKERS])
print(coef.groupby("country")[[*MODELS["main"], "r2", "r2_external"]].median().round(2).to_string())
print("Normal days per country:", nd.groupby("country").size().to_dict())

## Events
For every row of `events.csv` (main labels and alternative day 0s): AR and SAR = AR / s on every day of `AR_WIN`,
all outside the estimation window. A row eligible in 02 stays eligible if its window has `MIN_EST_DAYS` days and
AR(0) and AR(+1) can be computed. Outcome columns: `ar0`, `ar1`, `sar0`, `sar1` (and `ar0_ext`, `ar1_ext` of the
external-only model); `strong95` and `strong90` = 1 if |AR| of day 0 or day +1 is above the threshold of its day
class; `vol_pre` = ln(sd of AR over `PRE_VOL_WIN` / sd of AR over the country's normal days).

In [ ]:
outcomes, event_ar = event_outcomes(events, panels, fits["main"], normal["main"], thr)
ext = event_outcomes(events, panels, fits["external"], normal["external"])[0][["event_id", "variant", "ar0", "ar1"]]
outcomes = outcomes.merge(ext, on=["event_id", "variant"], how="left", suffixes=("", "_ext"), validate="one_to_one")
ok = outcomes["eligible"]
assert outcomes.loc[ok, ["vol_pre", "ar0_ext", "ar1_ext"]].notna().all().all()
here = ~ok & ~outcomes["drop_reason"].str.startswith("not eligible in 02")
print("Dropped here (not in 02):", dict(zip(outcomes.loc[here, "event_id"], outcomes.loc[here, "drop_reason"])))
print("Long-gap return on day 0 or +1 (kept):", outcomes.loc[ok & outcomes["long_gap"], "event_id"].tolist())
print(analysis_sample(events, outcomes).groupby("group").size().reindex(ANALYSIS_GROUPS).to_string())

## Outputs
`normal_days.csv` (AR and normal flag of both models), `thresholds.csv`, `event_outcomes.csv` (one row per row of
`events.csv`) and `event_ar.csv` (AR and SAR paths of the eligible rows, main model). The hash identifies the run.

In [ ]:
normal_out = normal["main"].assign(ar_ext=normal["external"]["ar"], normal_ext=normal["external"]["normal"])
for name, df in [("normal_days", normal_out), ("thresholds", thr), ("event_outcomes", outcomes), ("event_ar", event_ar)]:
    print(save_derived(df, name), len(df), "rows")
print("event_outcomes sha256:", hashlib.sha256(outcomes.to_csv(index=False).encode()).hexdigest()[:16])